# LightFM Model

### Unit 1 - 환경 및 공통 설정

In [ ]:
# Unit 1
import os
from pathlib import Path
import random

import numpy as np
import pandas as pd

RUNTIME = os.environ.get("LIGHTFM_RUNTIME", "local")
if RUNTIME != "linux-docker":
    raise RuntimeError(
        "공식 실행 환경이 아닙니다. ai/experiments 에서 "
        "'docker compose up' 후 JupyterLab에서 실행하세요."
    )

from lightfm import LightFM
from lightfm.data import Dataset
from lightfm.cross_validation import random_train_test_split
from lightfm.evaluation import precision_at_k, recall_at_k

SEED = int(os.environ.get("SEED", "42"))
TEST_RATIO = 0.2
EPOCHS = 30
NUM_THREADS = 2  # Docker/Linux OpenMP 환경 기준

random.seed(SEED)
np.random.seed(SEED)

DATA_FILES = {
    "review": Path("review_by_llm.csv"),
    "recipe": Path("recipe_fix.csv"),
    "ingredient_alias": Path("recipe_ingredient_alias.csv"),
}
ALLOWED_TARGET_MODES = (
    "star_sentiment_sum",
    "sentiment_only",
    "star_only",
    "ratio_1_2",
)
TARGET_MODE = os.environ.get("TARGET_MODE", "star_sentiment_sum")
if TARGET_MODE not in ALLOWED_TARGET_MODES:
    raise ValueError(f"TARGET_MODE must be one of {ALLOWED_TARGET_MODES}")
EXPORT_TAG = os.environ.get("EXPORT_TAG", TARGET_MODE)
GO_PRECISION_THRESHOLD = 0.05
if "EXCLUDED_RECIPE_COLUMNS" in os.environ:
    EXCLUDED_RECIPE_COLUMNS = [
        c for c in os.environ["EXCLUDED_RECIPE_COLUMNS"].split(",") if c
    ]
else:
    EXCLUDED_RECIPE_COLUMNS = ["ingredients"]  # 실험7 권장 피처 세트
if TARGET_MODE == "ratio_1_2":
    STAR_WEIGHT = float(os.environ.get("STAR_WEIGHT", "1.0"))
    SENTIMENT_WEIGHT = float(os.environ.get("SENTIMENT_WEIGHT", "2.0"))
else:
    STAR_WEIGHT = float(os.environ.get("STAR_WEIGHT", "1.0"))
    SENTIMENT_WEIGHT = float(os.environ.get("SENTIMENT_WEIGHT", "1.0"))
MODEL_MODE = "hybrid"
BASELINE_ONLY = os.environ.get("BASELINE_ONLY", "").strip().lower() in ("1", "true", "yes")

print({
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "epochs": EPOCHS,
    "target_mode": TARGET_MODE,
    "export_tag": EXPORT_TAG,
    "mode": MODEL_MODE,
    "excluded_recipe_columns": EXCLUDED_RECIPE_COLUMNS,
    "star_weight": STAR_WEIGHT,
    "sentiment_weight": SENTIMENT_WEIGHT,
    "baseline_only": BASELINE_ONLY,
})

### Unit 2 - 데이터 로드 및 필수 컬럼 검증

In [ ]:
# 데이터 프레임 로드
review_df = pd.read_csv(DATA_FILES["review"])
recipe_df = pd.read_csv(DATA_FILES["recipe"])
ingredient_alias_df = pd.read_csv(DATA_FILES["ingredient_alias"])

recipe_cols = [
    "RCP_SNO", "CKG_NM", "INQ_CNT", "SRAP_CNT", "CKG_MTH_ACTO_NM",
    "CKG_STA_ACTO_NM", "CKG_MTRL_ACTO_NM", "CKG_KND_ACTO_NM",
    "CKG_INBUN_NM", "CKG_DODF_NM", "CKG_TIME_NM",
]
ingredient_alias_cols = [
    "RCP_SNO", "CKG_NM", "ingredients_raw", "aliases_matched",
    "ingredients_normalized", "others_count", "others_items",
    "basic_count", "basic_items",
]
review_cols = [
    "recipe_id", "group_id", "star_count", "content",
    "positive", "negative", "star_norm",
]

for name, frame, cols in [
    ("recipe_fix.csv", recipe_df, recipe_cols),
    ("recipe_ingredient_alias.csv", ingredient_alias_df, ingredient_alias_cols),
    ("review_by_llm.csv", review_df, review_cols),
]:
    missing_cols = [c for c in cols if c not in frame.columns]
    if missing_cols:
        raise ValueError(f"Missing required columns in {name}: {missing_cols}")

recipe_df = recipe_df[recipe_cols].copy()
ingredient_alias_df = ingredient_alias_df[ingredient_alias_cols].copy()
review_df = review_df[review_cols].copy()

print("recipe_cols:", recipe_df.columns.tolist())
print("ingredient_alias_cols:", ingredient_alias_df.columns.tolist())
print("review_cols:", review_df.columns.tolist())

### Unit 3 - interaction 타겟 생성
- 사용자 데이터에 필요한 값들 가공 처리
- 레시피 데이터에 필요한 값들 가공 처리

> 유저 데이터 처리

In [ ]:
# 유저 데이터 전처리 - 별점 데이터 정리
# 강의 내용과 다르게 지금 수집한 내용 기준으로 정리, 필요한 경우 단계 나눠서 실험하면서 진행
# 값은 변환값 쓰지 않고 직접 계산해서 사용 
# star_count : 1 ~ 5점 데이터를 -1 ~ +1 사이 값으로 변환 ( -3 이후 * 1/2)
# review_df에서 별점 관련 데이터 드랍 및 필요한 칼럼만 남김

# 별점 데이터 관련 칼럼
star_related_cols = ["star_count", "star_norm"]

# 별점 값 가공해서 신규 컬럼으로 추가 (나중에 해당 부분에서 계산할 수 있도록)
# star_count 값을 -1 ~ +1 구간으로 변환해서 star 컬럼에 추가
# 변환식: ((star_count - 3) / 2)
review_df["star"] = review_df["star_count"].astype(float).apply(lambda x: (x - 3) / 2)


# review_df에서 별점 관련 칼럼 드랍
review_df = review_df.drop(columns=star_related_cols, errors="ignore")

# 필요한 값만 가공해서 사용: 여기서는 별점 관련 데이터가 사라지고
# 남은 칼럼만 유지됨을 확인
print("가공/드랍 이후 review_df 칼럼:", review_df.columns.tolist())

In [ ]:
# 유저 데이터 전처리 - 감성 데이터 정리 
# 긍정 - 부정 값을 감성분석 값으로 사용 (긍정인 경우 +, 부정인 경우 - 나오도록 처리)
# 컬럼은 sentiment 컬럼으로 추가하고 기존 감성분석 컬럼은 제거 

# 감성 데이터 전처리
# 'positive', 'negative' 값을 활용하여 sentiment 컬럼 생성
# sentiment = positive - negative (긍정은 +, 부정은 -)
review_df["sentiment"] = review_df["positive"].astype(float) - review_df["negative"].astype(float)

# 기존 감성 관련 컬럼(drop)
sentiment_related_cols = ["positive", "negative", "neutral", "compound"]
review_df = review_df.drop(columns=sentiment_related_cols, errors="ignore")

# 결과 칼럼 확인
print("감성 처리/드랍 이후 review_df 칼럼:", review_df.columns.tolist())

In [ ]:
# 리뷰 내용은 일단 제거 (content 컬럼)
# 리뷰 내용 제거 (content 컬럼 드랍)
review_df = review_df.drop(columns=["content"], errors="ignore")
print("content 컬럼 제거 후 review_df 칼럼:", review_df.columns.tolist())

> 레시피 데이터 처리

In [ ]:
# recipe_df, ingredient_alias_df를 RCP_SNO 기준으로 병합
# 중복 제거의 의미는 "중복 행"이 아니라 "겹치는 컬럼명" 처리(CKG_NM 등)

# recipe_df에 이미 있는 공통 컬럼은 ingredient_alias_df에서 제외하고 머지
recipe_base_cols = set(recipe_df.columns)
alias_merge_cols = [
    col for col in ingredient_alias_df.columns
    if col == "RCP_SNO" or col not in recipe_base_cols
]

ingredient_alias_for_merge = ingredient_alias_df[alias_merge_cols].copy()

recipe_df = recipe_df.merge(ingredient_alias_for_merge, on="RCP_SNO", how="left")

print("재료머지 이후 recipe_df 칼럼:", recipe_df.columns.tolist())
print(recipe_df.head())

In [ ]:
recipe_df.columns

In [ ]:
# 레시피 컬럼 개별 정리 
# 사용할 컬럼의 경우 컬럼 이름 수정 
# RCP_SNO -> recipe_id
# CKG_NM -> recipe_name
# INQ_CNT -> view_count
# SRAP_CNT -> scrap_count
# CKG_MTH_ACTO_NM -> cooking_method
# CKG_STA_ACTO_NM -> cooking_category
# CKG_MTRL_ACTO_NM -> main_ingred
# CKG_INBUN_NM -> dishes
# CKG_DODF_NM -> cooking_level
# CKG_TIME_NM -> cooking_time
# ingredients_raw -> 컬럼 드랍 (정규화 이름만 사용)
# aliases_matched -> aliases
# ingredients_normalized -> ingredients
# others_items -> 컬럼 드랍 (정규화된 재료 기준으로 유사도 비교)
# basic_items -> 컬럼 드랍 (기본 재료가 몇개 있는지만 체크)
# 위 기준으로 데이터 정리

# 컬럼명 매핑 정의
column_rename_map = {
    "RCP_SNO": "recipe_id",
    "CKG_NM": "recipe_name",
    "INQ_CNT": "view_count",
    "SRAP_CNT": "scrap_count",
    "CKG_MTH_ACTO_NM": "cooking_method",
    "CKG_STA_ACTO_NM": "cooking_category",
    "CKG_MTRL_ACTO_NM": "main_ingred",
    "CKG_KND_ACTO_NM": "recipe_kind",
    "CKG_INBUN_NM": "dishes",
    "CKG_DODF_NM": "cooking_level",
    "CKG_TIME_NM": "cooking_time",
    "aliases_matched": "aliases",
    "ingredients_normalized": "ingredients",
}

columns_to_drop = [
    "ingredients_raw",
    "others_items",
    "basic_items",
]

# 컬럼명 변경
recipe_df = recipe_df.rename(columns=column_rename_map)

# 필요없는 컬럼 드랍
recipe_df = recipe_df.drop(columns=[col for col in columns_to_drop if col in recipe_df.columns])

# 결과 컬럼 확인
print("정리 이후 recipe_df 컬럼:", recipe_df.columns.tolist())

In [ ]:
# Hybrid (아이템 피처 포함) 모드에서 alias_id, ingredients 정규화 처리

import ast

def str_to_list(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            return parsed if isinstance(parsed, list) else []
        except (ValueError, SyntaxError):
            return []
    return []

def normalize_aliases(values):
    raw_list = str_to_list(values)
    normalized = []
    for item in raw_list:
        if isinstance(item, dict):
            alias_id = item.get("alias_id")
            alias_name = item.get("name")
            token = alias_id if alias_id else alias_name
            if token:
                normalized.append(str(token))
        elif isinstance(item, str):
            normalized.append(item)
    return sorted(set(normalized))

def normalize_ingredients(values):
    raw_list = str_to_list(values)
    normalized = []
    for item in raw_list:
        # ingredients_normalized 형태: ["재료명", "수량", "단위"]
        if isinstance(item, list) and len(item) > 0:
            name = item[0]
            if name:
                normalized.append(str(name))
        elif isinstance(item, str):
            normalized.append(item)
    return sorted(set(normalized))

recipe_df["aliases"] = recipe_df["aliases"].apply(normalize_aliases)
recipe_df["ingredients"] = recipe_df["ingredients"].apply(normalize_ingredients)

print(recipe_df[["recipe_id", "aliases", "ingredients"]].head())

### Unit 4 - ID 매핑 및 Dataset 구성
- 내부 인덱스 매핑을 고정한다. (`group_id`, `recipe_id`)

In [ ]:
# 데이터 매트릭스로 정리하기 전 검사 
# 키 정합성: group_id, recipe_id null/빈값 제거, 문자열 캐스팅
# 중복 pair 처리: 같은 (group_id, recipe_id)가 여러 개면 집계 규칙 1개로 고정(평균/최대/최근)
# recipe_df와 review_df 각각 id 검사 및 매핑이 정상적으로 연결되는지 확인

# 1. Null/빈값 검사
missing_recipe_ids = recipe_df["recipe_id"].isnull().sum() + (recipe_df["recipe_id"].astype(str).str.strip() == "").sum()
missing_review_recipe_ids = review_df["recipe_id"].isnull().sum() + (review_df["recipe_id"].astype(str).str.strip() == "").sum()
missing_review_group_ids = review_df["group_id"].isnull().sum() + (review_df["group_id"].astype(str).str.strip() == "").sum()

print(f"[검사] recipe_df 'recipe_id' null 또는 빈값: {missing_recipe_ids}")
print(f"[검사] review_df 'recipe_id' null 또는 빈값: {missing_review_recipe_ids}")
print(f"[검사] review_df 'group_id' null 또는 빈값: {missing_review_group_ids}")

# 2. 연결 ID 정합성 확인
recipe_id_set = set(recipe_df["recipe_id"].astype(str).str.strip())
review_recipe_id_set = set(review_df["recipe_id"].astype(str).str.strip())
review_group_id_set = set(review_df["group_id"].astype(str).str.strip())

# review_df의 recipe_id가 recipe_df에 존재
unmatched_recipe_ids = review_recipe_id_set - recipe_id_set
print(f"[검사] review_df에서 recipe_df에 없는 recipe_id 개수: {len(unmatched_recipe_ids)}")
if len(unmatched_recipe_ids) > 0:
    print(f"(샘플) 미존재 recipe_id: {list(unmatched_recipe_ids)[:5]}")

# review_df의 group_id 및 recipe_id 조합 고유성 검사 (필요 시)
duplicated_pairs = review_df.duplicated(subset=["group_id", "recipe_id"], keep=False).sum()
print(f"[검사] review_df에서 (group_id, recipe_id) 중복 건수: {duplicated_pairs}")

# 정상적으로 연결되는지 확인 결과 샘플
matched_count = review_df[review_df["recipe_id"].astype(str).str.strip().isin(recipe_id_set)].shape[0]
print(f"[연결 확인] review_df에서 recipe_df로 매핑되는 review 개수: {matched_count} / {review_df.shape[0]}")

In [ ]:
# 위 검사에서 어느 한쪽이라도 매칭되지 않는 경우 해당 row 드랍
# recipe_df, review_df에서 recipe_id 및 group_id 매칭되지 않는 row 드랍
recipe_id_set = set(recipe_df["recipe_id"].astype(str).str.strip())
review_df = review_df[
    review_df["recipe_id"].astype(str).str.strip().isin(recipe_id_set)
]

# review_df의 group_id에서 빈값, null, str.strip()이 빈 칸인 row도 드랍
review_df = review_df[
    review_df["group_id"].notnull() & (review_df["group_id"].astype(str).str.strip() != "")
]

# recipe_df의 recipe_id도 빈값, null, str.strip()이 빈 칸인 row 드랍
recipe_df = recipe_df[
    recipe_df["recipe_id"].notnull() & (recipe_df["recipe_id"].astype(str).str.strip() != "")
]

In [ ]:
# 검사 후 매칭으로 누락되는 데이터 없는지 다시 확인

print(f"[최종] recipe_df shape: {recipe_df.shape}")
print(f"[최종] review_df shape: {review_df.shape}")

# recipe_id 기준으로 review_df에서 recipe_df에 없는 값이 남아있는지 확인
final_recipe_id_set = set(recipe_df["recipe_id"].astype(str).str.strip())
final_review_recipe_id_set = set(review_df["recipe_id"].astype(str).str.strip())
leftover_unmatched_recipe_ids = final_review_recipe_id_set - final_recipe_id_set
print(f"[최종검사] review_df에서 여전히 recipe_df에 없는 recipe_id 개수: {len(leftover_unmatched_recipe_ids)}")
if len(leftover_unmatched_recipe_ids) > 0:
    print(f"예시: {list(leftover_unmatched_recipe_ids)[:5]}")

# review_df에서 빈 group_id, 빈 recipe_id row 수 재확인
missing_group_id = review_df["group_id"].isnull().sum() + (review_df["group_id"].astype(str).str.strip() == "").sum()
missing_review_recipe_id = review_df["recipe_id"].isnull().sum() + (review_df["recipe_id"].astype(str).str.strip() == "").sum()
missing_recipe_id = recipe_df["recipe_id"].isnull().sum() + (recipe_df["recipe_id"].astype(str).str.strip() == "").sum()

print(f"[최종검사] review_df에서 group_id가 없는 row 수: {missing_group_id}")
print(f"[최종검사] review_df에서 recipe_id가 없는 row 수: {missing_review_recipe_id}")
print(f"[최종검사] recipe_df에서 recipe_id가 없는 row 수: {missing_recipe_id}")

In [ ]:
# Unit 4 - 정리한 데이터들을 데이터 셋으로 묶음 (Track B: 전 카탈로그 item)
CATALOG_USER_ID = "__catalog__"
user_ids = review_df["group_id"].astype(str).unique().tolist()
item_ids = recipe_df["recipe_id"].astype(str).tolist()
warm_item_ids = set(review_df["recipe_id"].astype(str).unique())
cold_item_ids = [i for i in item_ids if i not in warm_item_ids]

dataset = Dataset()
dataset.fit(users=user_ids + [CATALOG_USER_ID], items=item_ids)

print({
    "users": len(user_ids),
    "items": len(item_ids),
    "warm_items": len(warm_item_ids),
    "cold_items": len(cold_item_ids),
    "catalog_user": CATALOG_USER_ID,
})

### Unit 5 - interactions matrix 생성
- 학습/평가 입력 sparse matrix를 만든다.

In [ ]:
recipe_df.columns

In [ ]:
review_df.columns

In [ ]:
# 학습용 매트릭스를 만드는 부분

def calc_interaction_value(
    star,
    sentiment,
    *,
    target_mode,
    star_weight=1.0,
    sentiment_weight=1.0,
):
    if target_mode == "sentiment_only":
        return sentiment_weight * sentiment
    if target_mode == "star_only":
        return star_weight * star
    if target_mode == "ratio_1_2":
        return star_weight * star + sentiment_weight * sentiment
    return star_weight * star + sentiment_weight * sentiment


# Unit 5
star_s = pd.to_numeric(review_df["star"], errors="coerce").fillna(0.0)
sent_s = pd.to_numeric(review_df["sentiment"], errors="coerce").fillna(0.0)
review_df["interaction_value"] = calc_interaction_value(
    star_s,
    sent_s,
    target_mode=TARGET_MODE,
    star_weight=STAR_WEIGHT,
    sentiment_weight=SENTIMENT_WEIGHT,
)

triples = list(
    zip(
        review_df["group_id"].astype(str),
        review_df["recipe_id"].astype(str),
        review_df["interaction_value"].astype(float),
    )
)

interactions, _ = dataset.build_interactions(triples)
num_users, num_items = interactions.shape
density = interactions.nnz / (num_users * num_items) if num_users and num_items else 0.0

print({
    "target_mode": TARGET_MODE,
    "shape": interactions.shape,
    "nnz": interactions.nnz,
    "density": density,
})

### Unit 5b - item features (hybrid)
- 기본 제외: `ingredients` (실험7 권장). `EXCLUDED_RECIPE_COLUMNS` env로 override
- `view_count` / `scrap_count`는 항상 포함, feature token은 `log1p` 적용

In [ ]:
# Unit 5b
if BASELINE_ONLY:
    LOG_NUMERIC_COLUMNS = set()
    print("skip Unit 5b (BASELINE_ONLY)")
else:
    RECIPE_FEATURE_COLUMNS = [
        "recipe_name", "cooking_method", "cooking_category", "main_ingred",
        "dishes", "cooking_level", "cooking_time", "aliases", "ingredients",
        "recipe_kind", "others_count", "basic_count",
    ]
    FIXED_POPULARITY_COLUMNS = ["view_count", "scrap_count"]
    LOG_NUMERIC_COLUMNS = set(FIXED_POPULARITY_COLUMNS)
    CATEGORICAL_COLUMNS = [
        "recipe_name", "cooking_method", "cooking_category", "main_ingred",
        "dishes", "cooking_level", "cooking_time", "recipe_kind",
    ]
    NUMERIC_COLUMNS = ["others_count", "basic_count"]
    excluded = set(EXCLUDED_RECIPE_COLUMNS)

    def transform_numeric_feature(col, value):
        if pd.isna(value):
            return None
        v = max(0.0, float(value))
        if col in LOG_NUMERIC_COLUMNS:
            return f"{col}_log:{np.log1p(v):.4f}"
        return f"{col}:{int(v)}"

    def recipe_row_to_features(row):
        features = []
        for col in FIXED_POPULARITY_COLUMNS:
            token = transform_numeric_feature(col, row.get(col))
            if token:
                features.append(token)
        for col in CATEGORICAL_COLUMNS:
            if col in excluded:
                continue
            val = row.get(col)
            if pd.notna(val) and str(val).strip():
                features.append(f"{col}:{str(val).strip()}")
        for col in NUMERIC_COLUMNS:
            if col in excluded:
                continue
            token = transform_numeric_feature(col, row.get(col))
            if token:
                features.append(token)
        if "aliases" not in excluded:
            for token in row.get("aliases") or []:
                features.append(f"alias:{token}")
        if "ingredients" not in excluded:
            for token in row.get("ingredients") or []:
                features.append(f"ingredient:{token}")
        return features or ["recipe:unknown"]

    recipe_lookup = recipe_df.set_index(recipe_df["recipe_id"].astype(str))
    item_feature_tuples = []
    all_feature_names = set()
    for item_id in item_ids:
        if item_id in recipe_lookup.index:
            row = recipe_lookup.loc[item_id]
            feats = recipe_row_to_features(row)
        else:
            feats = ["recipe:unknown"]
        item_feature_tuples.append((item_id, feats))
        all_feature_names.update(feats)

    dataset.fit_partial(item_features=sorted(all_feature_names))
    item_features = dataset.build_item_features(item_feature_tuples)

    print({
        "item_feature_nnz": int(item_features.nnz),
        "unique_features": len(all_feature_names),
        "excluded_recipe_columns": sorted(excluded),
        "log_numeric_columns": sorted(LOG_NUMERIC_COLUMNS),
    })


### Unit 6 - train/test 분할

In [ ]:
# Unit 6
train, test = random_train_test_split(
    interactions,
    test_percentage=TEST_RATIO,
    random_state=np.random.RandomState(SEED),
)

print({"train_nnz": train.nnz, "test_nnz": test.nnz})

### Unit 10 - Random / train-popularity baseline
- LightFM과 동일 split·지표로 bar baseline 측정


In [ ]:
# Unit 10
import json
from pathlib import Path

from baseline_eval import evaluate_baseline, random_scores, train_popularity_scores

baseline_rng = np.random.default_rng(SEED)
random_baseline_metrics = evaluate_baseline(
    test, random_scores(interactions.shape[1], baseline_rng)
)
popularity_baseline_metrics = evaluate_baseline(test, train_popularity_scores(train))

baseline_report = {
    "experiment": "11_baseline",
    "seed": SEED,
    "test_ratio": TEST_RATIO,
    "baselines": {
        "random": random_baseline_metrics,
        "train_popularity": popularity_baseline_metrics,
    },
    "matrix": {
        "train_nnz": int(train.nnz),
        "test_nnz": int(test.nnz),
    },
}

runs_dir = Path("runs")
runs_dir.mkdir(exist_ok=True)
baseline_report_path = runs_dir / "baseline_report.json"
baseline_report_path.write_text(
    json.dumps(baseline_report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(baseline_report)
print(f"saved: {baseline_report_path}")


### Unit 7 - 학습

In [ ]:
# Unit 7
if BASELINE_ONLY:
    print("skip Unit 7-9 (BASELINE_ONLY)")
else:
    model = LightFM(loss="warp", random_state=SEED)
    precisions = []

    for epoch in range(EPOCHS):
        model.fit_partial(
            train, item_features=item_features, epochs=1, num_threads=NUM_THREADS
        )
        precision = precision_at_k(
            model, train, item_features=item_features, k=5
        ).mean()
        precisions.append(float(precision))
        print(f"Epoch {epoch+1}/{EPOCHS}, Precision@5: {precision:.4f}")


### Unit 8 - Precision/Recall 평가

In [ ]:
# Unit 8
if BASELINE_ONLY:
    pass
else:
    metrics = {
        "precision@5": float(
            precision_at_k(model, test, item_features=item_features, k=5).mean()
        ),
        "precision@10": float(
            precision_at_k(model, test, item_features=item_features, k=10).mean()
        ),
        "recall@5": float(
            recall_at_k(model, test, item_features=item_features, k=5).mean()
        ),
        "recall@10": float(
            recall_at_k(model, test, item_features=item_features, k=10).mean()
        ),
    }
    print(metrics)
    print({
        "vs_random_p5": metrics["precision@5"] - random_baseline_metrics["precision@5"],
        "vs_popularity_p5": metrics["precision@5"] - popularity_baseline_metrics["precision@5"],
    })


### Unit 11 - Track B catalog export
- 전 item `y_hat` 예측 → `recipe_lightfm.csv`
- warm 기준 선형 보정 `y_hat_linear` (관측 `review_rank_score`와 스케일 비교용)
- `catalog_eval.py`로 B0~B3·MAE/RMSE/R² 측정

In [ ]:
# Unit 11
if BASELINE_ONLY:
    track_b_eval = None
    print("skip Unit 11 (BASELINE_ONLY)")
else:
    from catalog_eval import (
        apply_linear_calibration,
        decomposed_track_b_metrics_df,
        evaluate_track_b,
        fit_linear_calibration,
        warm_obs_pred_metrics,
    )

    review_raw = pd.read_csv(DATA_FILES["review"])
    review_raw["sentiment_row"] = (
        review_raw["positive"].astype(float) - review_raw["negative"].astype(float)
    )
    review_agg = (
        review_raw.groupby("recipe_id", as_index=False)
        .agg(
            positive_avg=("positive", "mean"),
            negative_avg=("negative", "mean"),
            star_count_avg=("star_count", "mean"),
            star_norm_avg=("star_norm", "mean"),
            sentiment_avg=("sentiment_row", "mean"),
        )
        .assign(recipe_id=lambda d: d["recipe_id"].astype(str))
    )
    review_agg["review_rank_score"] = (
        review_agg["star_norm_avg"] + review_agg["sentiment_avg"]
    )

    export_df = recipe_df[["recipe_id", "recipe_name"]].copy()
    export_df["recipe_id"] = export_df["recipe_id"].astype(str)
    export_df = export_df.merge(review_agg, on="recipe_id", how="left")

    user_id_map, _, item_id_map, _ = dataset.mapping()
    catalog_user_idx = user_id_map[CATALOG_USER_ID]
    item_internal = np.array([item_id_map[i] for i in item_ids], dtype=np.int32)
    user_internal = np.full(len(item_ids), catalog_user_idx, dtype=np.int32)
    y_hat = model.predict(
        user_internal,
        item_internal,
        item_features=item_features,
        num_threads=NUM_THREADS,
    )
    export_df["y_hat"] = y_hat.astype(float)

    warm_mask = export_df["recipe_id"].isin(warm_item_ids).to_numpy()
    score_review = export_df["review_rank_score"].to_numpy(dtype=float)
    lin_slope, lin_intercept = fit_linear_calibration(
        export_df["y_hat"].to_numpy(), score_review, warm_mask
    )
    export_df["y_hat_linear"] = apply_linear_calibration(
        export_df["y_hat"].to_numpy(), lin_slope, lin_intercept
    )

    train_item_signal = np.asarray(train.sum(axis=0)).ravel().astype(float)
    track_b_eval = evaluate_track_b(
        export_df["y_hat"].to_numpy(),
        score_review,
        warm_mask,
        train_item_signal=train_item_signal,
    )
    track_b_eval.update(
        warm_obs_pred_metrics(
            export_df["y_hat"].to_numpy(), score_review, warm_mask
        )
    )
    track_b_eval["decomposed"] = decomposed_track_b_metrics_df(export_df, warm_item_ids)
    track_b_eval["target"] = "review_only"
    track_b_eval["score_review_formula"] = (
        "star_norm_avg + sentiment_avg (from review_by_llm)"
    )
    track_b_eval["y_train"] = TARGET_MODE
    track_b_eval["linear_formula"] = (
        f"review_rank_score ~= {lin_slope:.6f} * y_hat + {lin_intercept:.6f} (warm fit)"
    )
    track_b_eval["catalog_user"] = CATALOG_USER_ID
    export_csv_name = f"recipe_lightfm_exp14_{EXPORT_TAG}.csv"
    track_b_eval["export_csv"] = export_csv_name

    out_cols = [
        "recipe_id",
        "recipe_name",
        "positive_avg",
        "negative_avg",
        "star_count_avg",
        "star_norm_avg",
        "y_hat",
        "y_hat_linear",
        "review_rank_score",
    ]
    export_path = Path(export_csv_name)
    export_df[out_cols].to_csv(export_path, index=False, encoding="utf-8-sig")
    if EXPORT_TAG == "star_sentiment_sum" and TARGET_MODE == "star_sentiment_sum":
        legacy_path = Path("recipe_lightfm.csv")
        export_df[out_cols].to_csv(legacy_path, index=False, encoding="utf-8-sig")

    assert len(export_df) == len(recipe_df)
    assert np.isfinite(export_df["y_hat"]).all()
    assert np.isfinite(export_df["y_hat_linear"]).all()
    cold_mask = ~warm_mask
    if cold_mask.any():
        assert export_df.loc[cold_mask, "review_rank_score"].isna().all()
    if warm_mask.any():
        assert export_df.loc[warm_mask, "review_rank_score"].notna().all()

    print(track_b_eval)
    print(f"saved: {export_path.resolve()} ({len(export_df)} rows)")


### Unit 9 - 실험 리포트 기록

In [ ]:
# Unit 9
if BASELINE_ONLY:
    pass
else:
    import json

    go_no_go = metrics["precision@5"] >= GO_PRECISION_THRESHOLD
    experiment_id = (
        f"14_track_b_target_{EXPORT_TAG}"
        if os.environ.get("EXP14", "").strip().lower() in ("1", "true", "yes")
        else "13_track_b"
    )

    experiment_report = {
        "experiment": experiment_id,
        "data_files": {name: str(path) for name, path in DATA_FILES.items()},
        "mode": MODEL_MODE,
        "target_mode": TARGET_MODE,
        "export_tag": EXPORT_TAG,
        "star_weight": STAR_WEIGHT,
        "sentiment_weight": SENTIMENT_WEIGHT,
        "excluded_recipe_columns": EXCLUDED_RECIPE_COLUMNS,
        "seed": SEED,
        "test_ratio": TEST_RATIO,
        "epochs": EPOCHS,
        "loss": "warp",
        "log_numeric_columns": sorted(LOG_NUMERIC_COLUMNS),
        "matrix": {
            "num_users": int(interactions.shape[0]),
            "num_items": int(interactions.shape[1]),
            "nnz": int(interactions.nnz),
            "train_nnz": int(train.nnz),
            "test_nnz": int(test.nnz),
            "item_feature_nnz": int(item_features.nnz),
            "unique_features": len(all_feature_names),
            "warm_items": len(warm_item_ids),
            "cold_items": len(cold_item_ids),
        },
        "metrics": metrics,
        "track_b_eval": track_b_eval,
        "decision": {
            "go": go_no_go,
            "criterion": f"precision@5 >= {GO_PRECISION_THRESHOLD}",
        },
    }

    runs_dir = Path("runs")
    runs_dir.mkdir(exist_ok=True)
    report_path = runs_dir / "ablation_report.json"
    report_path.write_text(
        json.dumps(experiment_report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    exp14_path = runs_dir / f"exp14_{EXPORT_TAG}.json"
    exp14_path.write_text(
        json.dumps(experiment_report, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(experiment_report)
    print(f"saved: {report_path}")
    print(f"saved: {exp14_path}")
